In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
 sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
 sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_agreement_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AgreementId string, SourceSystem string,"
        #     "PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, AgreementKey string,"
        #     "AgreementStatusCode string, AgreementStatus string, IssueDate date,"
        #     "ExpiredDate date, SignedDate date, Currency string,"
        #     "LimitAmount decimal(19,2), LimitUtilizedAmount decimal(19,2),"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Agreement"

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "arotron",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== eaisig_cont ==================

        eaisig_cont_extra_filters = None

        if eaisig_cont_extra_filters is not None:
            eaisig_cont_extra_filters.append(date_filter)
            eaisig_cont_final_filters = " AND ".join(eaisig_cont_extra_filters)
        else:
            eaisig_cont_final_filters = date_filter

        eaisig_cont_selected_fields = [
            "eaisig_cont_case_s_id",
            "eaiSIG_CONT_CUST_S_UNID",
            "rowid",
            "eaisig_cont_case_status1",
            "eaisig_cont_case_status1_date",
            "eaisig_cont_case_opndate",
            "eaisig_cont_curr_code",
            "eaisig_cont_case_credlim",
            "eaisig_cont_case_spendlim",
            "businessdate",
        ]

        eaisig_cont = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaisig_cont",
            eaisig_cont_final_filters,
            eaisig_cont_selected_fields,
        )

        # =============== Join with portfolio ===============
        eaisig_cont = eaisig_cont.join(
            portfolio,
            on=substring(eaisig_cont["eaiSIG_CONT_CUST_S_UNID"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        eaisig_cont_primary_key_fields = [
            eaisig_cont.eaisig_cont_case_s_id,
            lit("arotron"),
        ]

        eaisig_cont = (
            eaisig_cont.withColumn("SourceSystem", lit("arotron"))
            .withColumn(
                "AgreementId",
                sha2(concat_ws(";", *eaisig_cont_primary_key_fields), 256),
            )
            .withColumn(
                "AgreementStatus",
                when((eaisig_cont.eaisig_cont_case_status1 == "1"), "Ενήμερο")
                .when((eaisig_cont.eaisig_cont_case_status1 == "2"), "Υπερήμερο")
                .when(
                    (eaisig_cont.eaisig_cont_case_status1 == "3"),
                    "Οριστική Καθυστέρηση",
                )
                .when((eaisig_cont.eaisig_cont_case_status1 == "4"), "Επισφάλεια")
                .otherwise(lit(None)),
            )
            .withColumn("ExpiredDate", lit(None).cast("date"))
            .withColumn("SignedDate", lit(None).cast("date"))
            .withColumn("IssueDate_date", col("eaisig_cont_case_opndate").cast("date"))
            .withColumn(
                "LimitAmount_decimal",
                col("eaisig_cont_case_credlim").cast("decimal(19,2)"),
            )
            .withColumn(
                "LimitUtilizedAmount_decimal",
                col("eaisig_cont_case_spendlim").cast("decimal(19,2)"),
            )
        )

        eaisig_cont = eaisig_cont.select(
            eaisig_cont.AgreementId,
            eaisig_cont.SourceSystem,
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            eaisig_cont.eaisig_cont_case_s_id.alias("AgreementKey"),
            eaisig_cont.eaisig_cont_case_status1.alias("AgreementStatusCode"),
            eaisig_cont.AgreementStatus.cast("string").alias("AgreementStatus"),
            eaisig_cont.IssueDate_date.alias("IssueDate"),
            eaisig_cont.ExpiredDate,
            eaisig_cont.SignedDate,
            eaisig_cont.eaisig_cont_curr_code.alias("Currency"),
            eaisig_cont.LimitAmount_decimal.alias("LimitAmount"),
            eaisig_cont.LimitUtilizedAmount_decimal.alias("LimitUtilizedAmount"),
            eaisig_cont.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        eaisig_cont.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_agreement_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
)